# Gabarito — Paralelismo

Este notebook traz a resolução comentada dos exercícios de [`0406a_paralelismo.ipynb`](0406a_paralelismo.ipynb).

⚠️ **Antes de olhar aqui:** tente resolver os exercícios sozinho no notebook original e use as células de verificação (`assert`) para conferir sua resposta. Volte a este gabarito só depois de tentar — ou se travar de verdade em algum passo.

In [1]:
# Imports necessários para os exercícios abaixo.
import math

import numpy as np


# Ferramentas do notebook principal (reconstruídas aqui).
def direcao(angulo_graus: float) -> np.ndarray:
    """Vetor de tamanho 1 apontando na direção dada, em graus, a partir da horizontal."""
    rad = math.radians(angulo_graus)
    return np.array([math.cos(rad), math.sin(rad)])


def medir_angulo(vertice, P, Q) -> float:
    """Transferidor digital: medida, em graus, do ângulo com vértice em `vertice` entre P e Q."""
    u = np.subtract(P, vertice)
    v = np.subtract(Q, vertice)
    cosseno = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    return math.degrees(math.acos(np.clip(cosseno, -1, 1)))


def cruzamento(P, inclinacao_1: float, Q, inclinacao_2: float):
    """Ponto em que a reta por P (inclinacao_1) cruza a reta por Q (inclinacao_2); None se paralelas."""
    matriz = np.column_stack([direcao(inclinacao_1), -direcao(inclinacao_2)])
    if abs(np.linalg.det(matriz)) < 1e-12:
        return None
    t, _ = np.linalg.solve(matriz, np.subtract(Q, P))
    return np.array(P, dtype=float) + t * direcao(inclinacao_1)

## Básico

**Exercício 1.** Os oito ângulos formados por uma transversal com $r \parallel s$, sabendo que o ângulo 1 mede 58°.

In [2]:
# Com r ∥ s, os oito ângulos só têm DUAS medidas: a do ângulo 1 e a do seu suplemento.
angulo_1 = 58
suplemento = 180 - angulo_1  # 1 e 2 são adjacentes e formam um ângulo raso

angulos_ex1 = {
    1: angulo_1,
    2: suplemento,
    3: angulo_1,    # oposto pelo vértice ao 1
    4: suplemento,  # oposto pelo vértice ao 2
    5: angulo_1,    # correspondente ao 1
    6: suplemento,  # correspondente ao 2 (ou alterno interno do 4)
    7: angulo_1,    # correspondente ao 3 (ou alterno externo do 1)
    8: suplemento,  # correspondente ao 4 (ou colateral externo do 1: 58 + 122 = 180)
}
print(angulos_ex1)
# Regra rápida: ímpares medem α, pares medem 180° − α.

{1: 58, 2: 122, 3: 58, 4: 122, 5: 58, 6: 122, 7: 58, 8: 122}


**Exercício 2.** Classificação de pares de ângulos e a relação entre eles.

In [3]:
# Para classificar, três perguntas: mesmo cruzamento? internos ou externos? mesmo lado de t?
# (a) 1 e 5: mesma posição (cima à direita) nos dois cruzamentos        -> correspondentes
# (b) 3 e 5: ambos internos, 3 à esquerda de t e 5 à direita            -> alternos internos
# (c) 2 e 8: ambos externos, 2 à esquerda e 8 à direita                 -> alternos externos
# (d) 4 e 5: ambos internos, ambos à direita de t                       -> colaterais internos
# (e) 2 e 7: ambos externos, ambos à esquerda de t                      -> colaterais externos
# (f) 6 e 8: mesmo cruzamento (t com s), "de costas" um para o outro    -> opostos pelo vértice
classificacao_ex2 = ["correspondentes", "alternos internos", "alternos externos",
                     "colaterais internos", "colaterais externos", "opostos pelo vértice"]

# Com r ∥ s: correspondentes, alternos e o.p.v. são congruentes; colaterais são suplementares.
# Conferindo com as medidas do Exercício 1: (d) 4 e 5 -> 122 + 58 = 180; (e) 2 e 7 -> 122 + 58 = 180.
relacao_ex2 = ["congruentes", "congruentes", "congruentes", "suplementares", "suplementares", "congruentes"]

print(classificacao_ex2)
print(relacao_ex2)

['correspondentes', 'alternos internos', 'alternos externos', 'colaterais internos', 'colaterais externos', 'opostos pelo vértice']
['congruentes', 'congruentes', 'congruentes', 'suplementares', 'suplementares', 'congruentes']


## Intermediário

**Exercício 3.** Função que decide, a partir de um par de ângulos, se as retas são paralelas.

In [4]:
def sao_paralelas_ex3(tipo: str, medida_1: float, medida_2: float) -> bool:
    """True se o par de ângulos (do tipo dado) garante que as retas cortadas são paralelas."""
    # Recíproca da seção 2:
    #   correspondentes, alternos internos e alternos externos -> paralelas se CONGRUENTES
    #   colaterais internos e colaterais externos             -> paralelas se SUPLEMENTARES
    if tipo.startswith("colaterais"):
        return math.isclose(medida_1 + medida_2, 180)
    return math.isclose(medida_1, medida_2)


print(sao_paralelas_ex3("alternos internos", 72, 72))     # True
print(sao_paralelas_ex3("colaterais internos", 60, 60))   # False: 60 + 60 = 120, não 180
print(sao_paralelas_ex3("colaterais externos", 90, 90))   # True: 90 + 90 = 180
# Repare no último: dois colaterais de 90° são, ao mesmo tempo, congruentes e suplementares.
# É o caso em que a transversal é perpendicular às duas retas (assunto de Perpendicularidade).

True
False
True


**Exercício 4.** A demonstração da seção 3, reproduzida em código.

In [5]:
def angulos_em_C_ex4(A, B, C) -> tuple[float, float, float]:
    """Os ângulos DĈA, AĈB e BĈE ao longo da paralela a AB traçada por C."""
    # O vetor B − A aponta na direção de AB. Somado a C (ou subtraído de C), dá pontos de uma
    # reta que passa por C com a MESMA direção de AB, ou seja, a paralela a AB por C.
    D = C - (B - A)  # do lado de A
    E = C + (B - A)  # do lado de B
    return medir_angulo(C, D, A), medir_angulo(C, A, B), medir_angulo(C, B, E)


triangulos_ex4 = [
    ((0, 0), (6, 0), (2, 4), "acutângulo"),
    ((0, 0), (6, 0), (0, 5), "retângulo em A"),
    ((0, 0), (6, 0), (-3, 2), "obtusângulo em A"),
    ((0, 0), (6, 0), (9, 1), "obtusângulo em B"),
    ((1, -2), (4, 5), (-3, 1), "girado e deslocado"),
    ((0, 0), (6, 0), (3, 0.2), "quase achatado"),
]
for *vertices, descricao in triangulos_ex4:
    A, B, C = (np.array(p, dtype=float) for p in vertices)
    dca, acb, bce = angulos_em_C_ex4(A, B, C)
    angulo_a, angulo_b = medir_angulo(A, B, C), medir_angulo(B, C, A)
    # Os três "assert" são os passos 2, 3 e 4-5 da tabela da demonstração
    assert math.isclose(dca, angulo_a)          # passo 2: DĈA = Â (alternos internos)
    assert math.isclose(bce, angulo_b)          # passo 3: BĈE = B̂ (alternos internos)
    assert math.isclose(dca + acb + bce, 180)   # passos 4 e 5: ângulo raso em C
    print(f"{descricao:>20}: Â = {angulo_a:6.2f}° = DĈA, B̂ = {angulo_b:6.2f}° = BĈE, "
          f"Ĉ = {acb:6.2f}°, soma = {dca + acb + bce:.2f}°")

          acutângulo: Â =  63.43° = DĈA, B̂ =  45.00° = BĈE, Ĉ =  71.57°, soma = 180.00°
      retângulo em A: Â =  90.00° = DĈA, B̂ =  39.81° = BĈE, Ĉ =  50.19°, soma = 180.00°
    obtusângulo em A: Â = 146.31° = DĈA, B̂ =  12.53° = BĈE, Ĉ =  21.16°, soma = 180.00°
    obtusângulo em B: Â =   6.34° = DĈA, B̂ = 161.57° = BĈE, Ĉ =  12.09°, soma = 180.00°
  girado e deslocado: Â =  76.33° = DĈA, B̂ =  37.06° = BĈE, Ĉ =  66.61°, soma = 180.00°
      quase achatado: Â =   3.81° = DĈA, B̂ =   3.81° = BĈE, Ĉ = 172.37°, soma = 180.00°


## Desafio

**Exercício 5.** Duas paralelas, duas transversais e uma cadeia de relações.

In [6]:
# (a) α (em Q₁, entre s para a esquerda e t₁ para cima) é o ângulo 6 da numeração da seção 2,
#     olhando t₁ como transversal. O correspondente dele em r (ângulo 2) fica em P₁, entre r
#     para a esquerda e t₁ para cima: correspondentes entre paralelas são congruentes.
angulo_a_ex5 = 115

# (b) Em P₁, esse ângulo e VP̂₁P₂ (entre r para a direita e t₁ para cima) são adjacentes e
#     formam um ângulo raso: suplementares.
angulo_VP1P2_ex5 = 180 - angulo_a_ex5  # 65

# (c) β (em Q₂, entre s para a direita e t₂ para cima) é o ângulo 5, olhando t₂ como transversal.
#     O ângulo em P₂ entre r para a direita e t₂ para baixo é o ângulo 4: ambos internos, ambos à
#     direita de t₂ -> colaterais internos, suplementares entre paralelas: 180 − 120 = 60.
#     VP̂₂P₁ (entre r para a esquerda e t₂ para cima) é o ângulo 2, oposto pelo vértice ao 4.
relacao_c_ex5 = "colaterais internos"
angulo_VP2P1_ex5 = 180 - 120  # 60

# (d) No triângulo VP₁P₂, a soma dos ângulos internos é 180° (agora demonstrada!).
x_ex5 = 180 - angulo_VP1P2_ex5 - angulo_VP2P1_ex5  # 55
print(angulo_a_ex5, angulo_VP1P2_ex5, relacao_c_ex5, angulo_VP2P1_ex5, x_ex5)

# Conferindo com as coordenadas da figura e o transferidor digital
Q1, Q2 = np.array([2.0, 0.0]), np.array([8.0, 0.0])
P1, P2 = cruzamento(Q1, 65, (0, 2.5), 0), cruzamento(Q2, 120, (0, 2.5), 0)
V = cruzamento(Q1, 65, Q2, 120)
print(f"VP̂₁P₂ = {medir_angulo(P1, V, P2):.2f}°, VP̂₂P₁ = {medir_angulo(P2, V, P1):.2f}°, "
      f"x = P₁V̂P₂ = {medir_angulo(V, P1, P2):.2f}°")

115 65 colaterais internos 60 55
VP̂₁P₂ = 65.00°, VP̂₂P₁ = 60.00°, x = P₁V̂P₂ = 55.00°


**Exercício 6.** O teorema do ângulo externo, demonstrado com uma paralela auxiliar.

Antes do código, a demonstração completa, no mesmo formato da seção 3 (é a Proposição 32 do Livro I dos *Elementos*):

| Passo | Afirmação | Justificativa |
|---|---|---|
| 1 | $\overleftrightarrow{CE} \parallel \overleftrightarrow{AB}$ | construção (postulado das paralelas) |
| 2 | $F\hat{C}E \equiv C\hat{A}B$, ou seja, $F\hat{C}E = \hat{A}$ | correspondentes: paralelas $\overleftrightarrow{CE}$ e $\overleftrightarrow{AB}$ cortadas pela transversal $\overleftrightarrow{AF}$ |
| 3 | $E\hat{C}B \equiv A\hat{B}C$, ou seja, $E\hat{C}B = \hat{B}$ | alternos internos: as mesmas paralelas cortadas pela transversal $\overleftrightarrow{BC}$ |
| 4 | $F\hat{C}B = F\hat{C}E + E\hat{C}B$ | a semirreta $\overrightarrow{CE}$ fica entre $\overrightarrow{CF}$ e $\overrightarrow{CB}$ (ângulos adjacentes) |
| 5 | $\text{ext}(C) = F\hat{C}B = \hat{A} + \hat{B}$ | substituindo 2 e 3 em 4 ∎ |

Repare que esta prova **não usa** a soma 180°: ela sai direto das paralelas. E, se quiser, dá para tirar o 180° dela: $\hat{C} + \text{ext}(C) = 180^\circ$ (suplementares), então $\hat{C} + \hat{A} + \hat{B} = 180^\circ$.

In [7]:
A6, B6, C6 = np.array([0.0, 0.0]), np.array([7.0, 0.0]), np.array([2.0, 4.0])

# (a) B − A aponta na direção de AB: somado a C, dá um ponto da paralela a AB por C.
#     C − A aponta de A para C: somado a C, continua o lado AC além de C.
E_ex6 = C6 + (B6 - A6)  # (9, 4)
F_ex6 = C6 + (C6 - A6)  # (4, 8)

# (b) O vértice (C) vai sempre no 1º argumento de medir_angulo.
angulo_FCE_ex6 = medir_angulo(C6, F_ex6, E_ex6)
angulo_ECB_ex6 = medir_angulo(C6, E_ex6, B6)

# (c) FĈE (em C) e CÂB (em A): mesma posição em relação à transversal AF, um "F" -> correspondentes.
#     EĈB (em C) e AB̂C (em B): entre as paralelas, em lados opostos da transversal BC, um "Z".
relacao_FCE_ex6 = "correspondentes"
relacao_ECB_ex6 = "alternos internos"

# (d) O ângulo externo é formado pelo prolongamento CF e pelo lado CB.
externo_ex6 = medir_angulo(C6, F_ex6, B6)
angulo_A6, angulo_B6 = medir_angulo(A6, B6, C6), medir_angulo(B6, C6, A6)
confere_ex6 = math.isclose(externo_ex6, angulo_A6 + angulo_B6)

print(f"E = {E_ex6}, F = {F_ex6}")
print(f"FĈE = {angulo_FCE_ex6:.4f}°  e  Â = {angulo_A6:.4f}°")
print(f"EĈB = {angulo_ECB_ex6:.4f}°  e  B̂ = {angulo_B6:.4f}°")
print(f"ext(C) = FĈB = {externo_ex6:.4f}°  e  Â + B̂ = {angulo_A6 + angulo_B6:.4f}°  ->  {confere_ex6}")

E = [9. 4.], F = [4. 8.]
FĈE = 63.4349°  e  Â = 63.4349°
EĈB = 38.6598°  e  B̂ = 38.6598°
ext(C) = FĈB = 102.0948°  e  Â + B̂ = 102.0948°  ->  True
